In [ ]:
# -*- coding: utf-8 -*-
import os, time, numpy as np, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from pathlib import Path
import os

# Find repo root robustly whether execution starts in repo root or in code/
HERE = Path.cwd()

if (HERE / "data").exists() and (HERE / "results").exists():
    REPO_ROOT = HERE
elif (HERE.parent / "data").exists() and (HERE.parent / "results").exists():
    REPO_ROOT = HERE.parent
else:
    raise FileNotFoundError(
        f"Could not locate repo root from working directory: {HERE}"
    )

print("Working directory:", HERE)
print("Resolved repo root:", REPO_ROOT)
# ---- GPU setup (run every session once) ----
for g in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception as e:
        print("Could not set memory growth:", e)
print("Devices:", tf.config.list_physical_devices())

# ---- Config ----
DATA_PATH = REPO_ROOT / "data"
OUT_CSV = REPO_ROOT / "results" / "intermediate" / "nn_table4.csv"

os.makedirs(REPO_ROOT / "results" / "intermediate", exist_ok=True)

TURBINES = list(range(1, 67))
TESTSET  = [38,39,40,41,42,43,44] 
TRAINSET = [i for i in TURBINES if i not in TESTSET]


FEATURES = ["wind_speed", "temperature"]
TARGET   = "power"

# ---- ANN hyperparameters ----
UNITS1, UNITS2, UNITS3 = 8, 16, 8
LR, EPOCHS, BATCH_SIZE, SEED = 0.001, 100, 2048, 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.makedirs("results", exist_ok=True)

def build_model(seed=SEED):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    m = Sequential([
        layers.Input(shape=(len(FEATURES),)),
        layers.Dense(UNITS1, activation='relu'),
        layers.Dense(UNITS2, activation='relu'),
        layers.Dense(UNITS3, activation='relu'),
        layers.Dense(1)
    ])
    m.compile(
        loss=keras.losses.MeanSquaredError(),
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        metrics=['mae', 'mse']
    )
    return m

def read_csv_safe(path):
    path = Path(path)
    try:
        if path.exists():
            return pd.read_csv(path)
    except Exception:
        pass
    return None

# =========================================================
# Build ONE training set from all 2017 data in TRAINSET
# =========================================================
train_parts = []

for j in TRAINSET:
    tr = read_csv_safe(DATA_PATH / f"Turbine{j}_2017.csv")
    if tr is None or not set(FEATURES + [TARGET]).issubset(tr.columns):
        continue

    tr = tr[FEATURES + [TARGET]].dropna()
    if tr.empty:
        continue

    train_parts.append(tr)

if len(train_parts) == 0:
    raise ValueError("No valid training data found in TRAINSET.")

train_all = pd.concat(train_parts, axis=0, ignore_index=True)

X_train = train_all[FEATURES].to_numpy(np.float32)
y_train = train_all[TARGET].to_numpy(np.float32).reshape(-1)

# ---- Scale ----
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)

# ---- Train model once ----
t1 = time.time()
model = build_model()
model.fit(
    X_train_s, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
    shuffle=True
)
t2 = time.time()
train_time = round(t2 - t1, 2)

# =========================================================
# Predict on each turbine in TESTSET (2017 only)
# =========================================================
rows = []

for i in TESTSET:
    print(f"\n=== Evaluating Turbine {i} on 2017 ===")

    test17 = read_csv_safe(DATA_PATH / f"Turbine{i}_2017.csv")
    if test17 is None or not set(FEATURES + [TARGET]).issubset(test17.columns):
        print(f"Skipping Turbine {i} (missing 2017 data or columns)")
        continue

    test17 = test17[FEATURES + [TARGET]].dropna()
    if test17.empty:
        print(f"Skipping Turbine {i} (empty after dropna)")
        continue

    X_te17 = test17[FEATURES].to_numpy(np.float32)
    y_te17 = test17[TARGET].to_numpy(np.float32).reshape(-1)
    X_te17_s = scaler.transform(X_te17)

    p1 = time.time()
    pred17 = model.predict(X_te17_s, batch_size=BATCH_SIZE, verbose=0).reshape(-1)
    pred17 = np.clip(pred17, 0.0, None)
    p2 = time.time()

    pred17_time = round(p2 - p1, 2)
    rmse17 = float(np.sqrt(mean_squared_error(y_te17, pred17)))

    rows.append({
        "Turbine": i,
        "Method": "NN",
        "RMSE_2017": rmse17,
        "Runtime_train": train_time,
        "Runtime_pred17": pred17_time
    })

    print(pd.DataFrame([rows[-1]]))

# ---- Save ----
results = pd.DataFrame(
    rows,
    columns=["Turbine", "Method", "RMSE_2017", "Runtime_train", "Runtime_pred17"]
)

results.to_csv(OUT_CSV, index=False)
print("\nSaved:", OUT_CSV)
print(results.head())


# ---- Update final results.csv : Table 4, Multi-layer NN row ----
from pathlib import Path
import importlib.util

upd_path = REPO_ROOT / "code" / "update_final_results.py"
spec = importlib.util.spec_from_file_location("update_final_results", upd_path)
upd_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(upd_mod)

update_final_results = upd_mod.update_final_results

nn_table4_rmse = results["RMSE_2017"].mean() if not results.empty else np.nan

update_final_results(
    method="Multi-layer NN",
    table_id="Table 4",
    rmse=nn_table4_rmse
)